# H1 · Interpolation error on degraded CIRRUS

Issue [#18](https://github.com/fedro86/DECADE/issues/18). CIRRUS (1 km / 5 min) is degraded as if it were
ODYSSEY: mean in linear Z over 2 × 2 blocks (`opera_ds.validate.degrade_frame`), one frame every 15 min.
The degraded frames are interpolated back to 1 km / 5 min with the methods of `opera_ds.interp` and
compared with the real CIRRUS. `run_validation` runs unchanged: only its reader of the coarse files is
`read_degraded`. The truth is known, so the error is the interpolation error alone, without the
ODYSSEY–CIRRUS product difference (H3).

Choices agreed in #65:

- **time shift:** none in the main runs. The degraded frame at NT is CIRRUS at NT, so `alpha` = 0 is a
  real (degraded) frame;
- **precipitation type:** per pixel, with Steiner et al. (1995) applied to the truth
  (`opera_ds.classify.obs_classes`); the scores are also accumulated on the convective and on the
  stratiform pixels (methods `<method>|convective`, `<method>|stratiform`). Rule and sources:
  `docs/method.md`, `docs/sources/`.

In [ ]:
%load_ext autoreload
%autoreload 2
from opera_ds.paths import DATA_DIR, RESULTS_DIR

import csv
import datetime as dt
import functools
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np

from opera_ds.classify import obs_classes
from opera_ds.odim_io import index_files, read_odim_composite
from opera_ds.paths import REPO_DIR
from opera_ds.validate import read_degraded, run_validation

## Parameters

In [ ]:
START = dt.datetime(2025, 9, 8)        # one week for the prototype (rainy, with convection)
END = dt.datetime(2025, 9, 15)         # excluded
METHODS = ("nearest", "linear", "lk")
WORKERS = 12
OUT = RESULTS_DIR / "h1_degraded_cirrus" / f"{START:%Y%m%d}-{END:%Y%m%d}"
OUT.mkdir(parents=True, exist_ok=True)

## Files and Belgium mask

In [ ]:
cirrus = index_files(str(DATA_DIR / "cirrus_bel" / "**" / "*@DBZH.h5"), r"@(\d{8}T\d{4})@", "%Y%m%dT%H%M")
margin = dt.timedelta(hours=1)         # motion context around the period
cirrus = {t: p for t, p in cirrus.items() if START - margin <= t < END + margin}
coarse = {t: p for t, p in cirrus.items() if t.minute % 15 == 0}
print(f"CIRRUS frames: {len(cirrus)}, degraded frames (every 15 min): {len(coarse)}")

ref = read_odim_composite(next(iter(cirrus.values())))
r0, c0 = int(ref.where["crop_row0"]), int(ref.where["crop_col0"])
ny, nx = ref.values.shape
region = np.load(DATA_DIR / "mask_BEL_1km.npy")[r0:r0 + ny, c0:c0 + nx]
np.save(OUT / "region_mask_crop.npy", region)
print(f"crop {ny} x {nx} px at 1 km; Belgium {region.sum()} px")

## Which degradation looks like ODYSSEY?

Review of #65: if ODYSSEY does not make its 2 km pixel as a mean in Z, CIRRUS degraded by a mean
could be smoother (easier to interpolate) than real ODYSSEY. On the overlap days 1–5/7/2024 three
degradations of CIRRUS (`zmean`, `max`, `sample`) are compared with real ODYSSEY: distribution of wet
pixels after removing the median offset (the products differ in level, H3) and small-scale energy.
ODYSSEY at NT is paired with CIRRUS at NT+5, the closest window (`docs/data.md`, *Time windows*):
CIRRUS has no frame at the quarter hours on those days. Uses the whole files downloaded for S0.

In [ ]:
import cv2

from opera_ds.odim_io import bbox_extent
from opera_ds.validate import DEGRADE_HOWS, degrade_frame

CAP, WET = 55.0, 10.0                  # dBZ: cap against ODYSSEY outliers (docs/data.md); wet pixels as in the validation
BINS = np.arange(-10.0, 75.5, 0.5)
QS = np.linspace(0.05, 0.99, 95)
ody24 = index_files(str(DATA_DIR / "odyssey" / "*@DBZH_QIND.h5"), r"@(\d{8}T\d{4})@", "%Y%m%dT%H%M")
cir24 = index_files(str(DATA_DIR / "cirrus" / "*@DBZH.h5"), r"@(\d{8}T\d{4})@", "%Y%m%dT%H%M")
ext24 = bbox_extent(read_odim_composite(next(iter(ody24.values()))).where, (2.5, 49.45, 6.45, 51.55), 200)
_c = read_odim_composite(next(iter(cir24.values())), extent=ext24)
_r0, _c0 = int(_c.where["crop_row0"]), int(_c.where["crop_col0"])
_ny, _nx = _c.values.shape
bel2 = (np.load(DATA_DIR / "mask_BEL_1km.npy")[_r0:_r0 + _ny, _c0:_c0 + _nx]
        .reshape(_ny // 2, 2, _nx // 2, 2).all(axis=(1, 3)))


def hp_energy(F, sel, sigma=2.0):
    """High-pass energy (sum of squares) of F inside sel, as in the validation scores."""
    Fz = np.where(sel, F, 0).astype(np.float32)
    num = cv2.GaussianBlur(Fz, (0, 0), sigma)
    den = cv2.GaussianBlur(sel.astype(np.float32), (0, 0), sigma)
    ok = sel & (den > 0.5)
    return float(((Fz - num / np.maximum(den, 1e-6))[ok] ** 2).sum()), int(ok.sum())


names = ("ODYSSEY",) + DEGRADE_HOWS
stats = {a: {k: dict(hist=np.zeros(len(BINS) - 1), e=0.0, n=0) for k in names} for a in ("Belgium + 200 km", "Belgium")}
n_pairs = 0
for t, p in ody24.items():
    tc = t + dt.timedelta(minutes=5)
    if tc not in cir24:
        continue
    fo = read_odim_composite(p, extent=ext24)
    fc = read_odim_composite(cir24[tc], extent=ext24)
    frames = {"ODYSSEY": fo, **{h: degrade_frame(fc, 2, h) for h in DEGRADE_HOWS}}
    cov = fo.coverage & frames["zmean"].coverage
    for area, sel in (("Belgium + 200 km", cov), ("Belgium", cov & bel2)):
        for k, fr in frames.items():
            F = np.minimum(np.nan_to_num(fr.to_field(0.0)), CAP)
            s = stats[area][k]
            s["hist"] += np.histogram(F[sel & (F >= WET)], BINS)[0]
            e, m = hp_energy(F, sel)
            s["e"] += e
            s["n"] += m
    n_pairs += 1
print(f"{n_pairs} pairs ODYSSEY NT / CIRRUS NT+5")

In [ ]:
centers = 0.5 * (BINS[1:] + BINS[:-1])

def quantiles(h, q):
    return np.interp(q, np.cumsum(h) / h.sum(), centers)

print(f"{'area':17s} {'variant':8s} {'offset':>7s} {'QQ shape RMS':>12s} {'energy ratio':>12s}")
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, (area, st) in zip(axes, stats.items()):
    med_o = quantiles(st["ODYSSEY"]["hist"], 0.5)
    q_o = quantiles(st["ODYSSEY"]["hist"], QS) - med_o
    e_o = st["ODYSSEY"]["e"] / st["ODYSSEY"]["n"]
    for k in DEGRADE_HOWS:
        med = quantiles(st[k]["hist"], 0.5)
        q = quantiles(st[k]["hist"], QS) - med
        rms = np.sqrt(np.mean((q - q_o) ** 2))
        print(f"{area:17s} {k:8s} {med - med_o:7.2f} {rms:12.2f} {st[k]['e'] / st[k]['n'] / e_o:12.2f}")
        ax.plot(q_o, q, label=k)
    lim = [q_o.min(), q_o.max()]
    ax.plot(lim, lim, "k--", linewidth=1)
    ax.set_title(f"{area}: wet-pixel quantiles minus median")
    ax.set_xlabel("ODYSSEY [dB]")
    ax.set_ylabel("degraded CIRRUS [dB]")
    ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "h1_degraded_cirrus" / "degradation_vs_odyssey_qq.png", dpi=130)

## Validation against the real CIRRUS

In [ ]:
t = dt.datetime.now()
run_validation(coarse, cirrus, START, END, str(OUT), shifts=(0.0,), methods=METHODS,
               pair_stride=1, workers=WORKERS, region_mask_path=str(OUT / "region_mask_crop.npy"),
               reader=functools.partial(read_degraded, factor=2), cirrus_reader=read_odim_composite,
               classifier=obs_classes)
print(f"done in {(dt.datetime.now() - t).total_seconds() / 60:.1f} min; outputs in {OUT}")

In [ ]:
def read_csv(path):
    with open(path, encoding="utf-8") as f:
        return list(csv.DictReader(f))

summary_all = read_csv(OUT / "summary.csv")
summary = [r for r in summary_all if "|" not in r["method"]]
cols =("rmse", "bias", "corr", "csi_20", "csi_30", "fss_20dBZ_11km", "hf_energy_ratio")
print(f"{'method':8s} {'alpha':>6s} {'times':>6s} " + " ".join(f"{c:>15s}" for c in cols))
for r in sorted(summary, key=lambda r: (r["method"], float(r["alpha"]))):
    print(f"{r['method']:8s} {float(r['alpha']):6.3f} {int(r['n_times']):6d} "
          + " ".join(f"{float(r[c]):15.3f}" for c in cols))

## By `alpha`, against the 4 days of real ODYSSEY

`alpha` = 0 is a real frame: its error is the floor (upscaling only, here). The extra error at
`alpha` = 1/3 and 2/3 is the interpolation error. RMSEs do not subtract: with independent errors they
add in quadrature, so the extra error is √(RMSE_α² − RMSE_0²) at the mean of α = 1/3 and 2/3, with
the ratio RMSE_α / RMSE_0 next to it. Both runs are taken at shift 0, so the `alpha` values match. Real ODYSSEY has no `alpha` = 0: on 1–5/7/2024 CIRRUS is missing at the quarter
hours (`docs/data.md`, *Overlap between products*).

In [ ]:
ref4 = [r for r in read_csv(REPO_DIR / "docs" / "figures" / "summary.csv") if float(r["shift_min"]) == 0.0]

def rmse_by_alpha(rows, m):
    return {round(float(r["alpha"]), 3): float(r["rmse"]) for r in rows if r["method"] == m}

def extra_error(a0, a1, a2):
    """Interpolation error in quadrature over the floor, and the ratio, at the mean of a1, a2."""
    mid = np.sqrt((a1 ** 2 + a2 ** 2) / 2)
    return np.sqrt(max(mid ** 2 - a0 ** 2, 0.0)), mid / a0

print(f"{'method':8s} {'':22s} {'a=0':>7s} {'a=1/3':>7s} {'a=2/3':>7s} {'extra':>7s} {'ratio':>7s}")
for m in METHODS:
    for name, rows in (("degraded CIRRUS", summary), ("real ODYSSEY, 4 days", ref4)):
        d = rmse_by_alpha(rows, m)
        a0, a1, a2 = (d.get(a, np.nan) for a in (0.0, 0.333, 0.667))
        extra, ratio = extra_error(a0, a1, a2)
        print(f"{m:8s} {name:22s} {a0:7.2f} {a1:7.2f} {a2:7.2f} {extra:7.2f} {ratio:7.2f}")

## By scale (interpolated times)

In [ ]:
fss_cols = [c for c in summary[0] if c.startswith("fss_20dBZ_")]
r_cols = [c for c in summary[0] if c.startswith("r") and c.endswith("_rmse")]
print(f"{'method':8s} {'alpha':>6s} " + " ".join(f"{c[6:]:>8s}" for c in fss_cols) + " | "
      + " ".join(f"{c:>9s}" for c in r_cols))
for r in sorted(summary, key=lambda r: (r["method"], float(r["alpha"]))):
    print(f"{r['method']:8s} {float(r['alpha']):6.3f} " + " ".join(f"{float(r[c]):8.3f}" for c in fss_cols)
          + " | " + " ".join(f"{float(r[c]):9.3f}" for c in r_cols))
print("FSS at 20 dBZ by window (km); RMSE of R (mm/h) by block (km)")

## By precipitation type (Steiner pixels)

Scores on the convective and on the stratiform pixels of the truth, classified with Steiner et al.
(1995). No per-time label is needed.

In [ ]:
cls_rows = {(r["method"], round(float(r["alpha"]), 3)): r for r in summary_all}
print(f"{'method':8s} {'class':11s} {'pixels a=0':>11s} {'a=0':>7s} {'a=1/3':>7s} {'a=2/3':>7s} {'extra':>7s} {'ratio':>7s} {'CSI30 a=1/3':>11s}")
for m in METHODS:
    for k in ("convective", "stratiform"):
        rr = [cls_rows.get((f"{m}|{k}", a)) for a in (0.0, 0.333, 0.667)]
        a0, a1, a2 = (float(r["rmse"]) for r in rr)
        extra, ratio = extra_error(a0, a1, a2)
        print(f"{m:8s} {k:11s} {int(rr[0]['n_wet_px']):11d} {a0:7.2f} {a1:7.2f} {a2:7.2f} {extra:7.2f} {ratio:7.2f} {float(rr[1]['csi_30']):11.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, k in zip(axes, ("convective", "stratiform")):
    for m in METHODS:
        a = [0.0, 0.333, 0.667]
        ax.plot([x * 15 for x in a], [float(cls_rows[(f"{m}|{k}", x)]["rmse"]) for x in a], marker="o", label=m)
    ax.set_title(f"{k} pixels (Steiner)")
    ax.set_xlabel("minutes after degraded frame t0")
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("RMSE on wet pixels [dBZ]")
axes[0].legend()
fig.tight_layout()
fig.savefig(OUT / "rmse_by_precipitation_type.png", dpi=130)

## Convective area per time (for an event label)

Area of convective pixels over Belgium at each truth time, the label for D3 and for reporting. The
threshold is not chosen here: it must make the D3 events come out convective and most of January not.

In [ ]:
times = sorted({dt.datetime.fromisoformat(r["time"]) for r in read_csv(OUT / "per_time.csv")})
area = []
for t in times:
    obs = read_odim_composite(cirrus[t]).to_field(0.0)
    area.append((obs_classes(obs)["convective"] & region).sum())   # 1 km pixels = km2
area = np.array(area, float)
print(f"{len(area)} wet times; convective area over Belgium [km2]: "
      f"median {np.median(area):.0f}, p75 {np.percentile(area, 75):.0f}, p90 {np.percentile(area, 90):.0f}, max {area.max():.0f}")
np.save(OUT / "convective_area_km2.npy", area)

## Main runs: all months

The same validation month by month (January, June, July, September, October 2025: three seasons, as
agreed in #65). Each month takes about 45 min on the workstation; the cell skips months already done.

In [ ]:
RUN_MONTHS = ["2025-01", "2025-06", "2025-07", "2025-09", "2025-10"]   # 3+ months, two seasons (#18, #65)


def run_month(ym):
    """Main H1 run for one month, same settings as above; skipped if its DONE file exists."""
    a = dt.datetime.strptime(ym, "%Y-%m")
    b = (a + dt.timedelta(days=32)).replace(day=1)
    out = RESULTS_DIR / "h1_degraded_cirrus" / f"{a:%Y%m%d}-{b:%Y%m%d}"
    if (out / "DONE").exists():
        return out
    out.mkdir(parents=True, exist_ok=True)
    allfiles = index_files(str(DATA_DIR / "cirrus_bel" / "**" / "*@DBZH.h5"), r"@(\d{8}T\d{4})@", "%Y%m%dT%H%M")
    cir = {t: p for t, p in allfiles.items() if a - margin <= t < b + margin}
    crs = {t: p for t, p in cir.items() if t.minute % 15 == 0}
    np.save(out / "region_mask_crop.npy", region)
    t0 = dt.datetime.now()
    run_validation(crs, cir, a, b, str(out), shifts=(0.0,), methods=METHODS, pair_stride=1,
                   workers=WORKERS, region_mask_path=str(out / "region_mask_crop.npy"),
                   reader=functools.partial(read_degraded, factor=2), cirrus_reader=read_odim_composite,
                   classifier=obs_classes)
    (out / "DONE").write_text(f"{(dt.datetime.now() - t0).total_seconds() / 60:.1f} min\n", encoding="utf-8")
    return out


# about 45 min per month on the workstation (12 processes); set RUN = True to (re)run missing months
RUN = False
if RUN:
    for ym in RUN_MONTHS:
        run_month(ym)

## By month and by season

RMSE pooled over months weighted by the wet pixels; seasons DJF, MAM, JJA, SON as in Imhoff et al. (2020)
(`docs/sources/2020 - Imhoff - …md`), the coarse second split asked in #65. Classes are Steiner at 2 km.

In [ ]:
SEASON = {12: "DJF", 1: "DJF", 2: "DJF", 3: "MAM", 4: "MAM", 5: "MAM",
          6: "JJA", 7: "JJA", 8: "JJA", 9: "SON", 10: "SON", 11: "SON"}   # seasons as in Imhoff et al. (2020)
ROWS = ["nearest", "linear", "lk", "nearest|convective", "linear|convective", "lk|convective",
        "nearest|stratiform", "linear|stratiform", "lk|stratiform"]
month_dirs = sorted(p for p in (RESULTS_DIR / "h1_degraded_cirrus").glob("2025??01-20????01") if (p / "DONE").exists())
print("months available:", [p.name[:6] for p in month_dirs])


def load(p):
    with open(p / "summary.csv", encoding="utf-8") as f:
        return {(r["method"], round(float(r["alpha"]), 3)): (int(r["n_wet_px"]), float(r["rmse"]))
                for r in csv.DictReader(f)}


def pooled(tables, m, a):
    """RMSE pooled over months, weighted by the number of wet pixels."""
    n = sum(t[(m, a)][0] for t in tables if (m, a) in t)
    return np.sqrt(sum(t[(m, a)][0] * t[(m, a)][1] ** 2 for t in tables if (m, a) in t) / n) if n else np.nan


def show(name, tables):
    print(f"== {name}")
    print(f"{'method':20s} {'a=0':>6s} {'a=1/3':>6s} {'a=2/3':>6s} {'extra':>6s} {'ratio':>6s}")
    for m in ROWS:
        a0, a1, a2 = (pooled(tables, m, a) for a in (0.0, 0.333, 0.667))
        mid = np.sqrt((a1 ** 2 + a2 ** 2) / 2)
        print(f"{m:20s} {a0:6.2f} {a1:6.2f} {a2:6.2f} {np.sqrt(max(mid ** 2 - a0 ** 2, 0.0)):6.2f} {mid / a0:6.2f}")


tables = {p.name[:6]: load(p) for p in month_dirs}
for ym, t in tables.items():
    show(ym, [t])
by_season = {}
for ym, t in tables.items():
    by_season.setdefault(SEASON[int(ym[4:6])], []).append(t)
for s, ts in by_season.items():
    show(f"season {s} ({len(ts)} months)", ts)
if tables:
    show("all months", list(tables.values()))

## By scale

Scores by scale asked in #18 and in the review of #65: FSS at 20 and 30 dBZ by window size (`DEFAULT_FSS_SCALES`,
1-81 km) and RMSE of R on blocks of 1-25 km (`DEFAULT_R_SCALES`), all pixels, `alpha` = 0 and 1/3 (2/3 is close to
1/3; the largest difference is printed).

In [ ]:
def load_rows(p):
    with open(p / "summary.csv", encoding="utf-8") as f:
        return {(r["method"], round(float(r["alpha"]), 3)): r for r in csv.DictReader(f)}


def scale_table(cols, label, digits=3):
    """Markdown table by month: alpha = 0 (floor, the same for all methods) and each method at alpha = 1/3."""
    head = [c.split("_")[-1] if c.startswith("fss_") else c.split("_")[0][1:] + " km" for c in cols]
    lines = [f"| month | method, `alpha` | " + " | ".join(head) + " |",
             "|---|---|" + "---:|" * len(cols)]
    for p in month_dirs:
        rows = load_rows(p)
        name = dt.datetime.strptime(p.name[:6], "%Y%m").strftime("%B %Y")
        lines.append(f"| {name} | all, 0 | " + " | ".join(f"{float(rows[('lk', 0.0)][c]):.{digits}f}" for c in cols) + " |")
        for m in ("nearest", "linear", "lk"):
            lines.append(f"| | `{m}`, 1/3 | " + " | ".join(f"{float(rows[(m, 0.333)][c]):.{digits}f}" for c in cols) + " |")
    print(f"**{label}**\n")
    print("\n".join(lines) + "\n")
    # alpha = 2/3 against alpha = 1/3
    d = max(abs(float(load_rows(p)[(m, 0.333)][c]) - float(load_rows(p)[(m, 0.667)][c]))
            for p in month_dirs for m in ("nearest", "linear", "lk") for c in cols)
    print(f"largest difference between alpha = 1/3 and 2/3: {d:.3f}\n")


cols0 = load_rows(month_dirs[0])[("lk", 0.0)]
scale_table([c for c in cols0 if c.startswith("fss_20dBZ_")], "FSS at 20 dBZ by window size")
scale_table([c for c in cols0 if c.startswith("fss_30dBZ_")], "FSS at 30 dBZ by window size")
scale_table([c for c in cols0 if c.startswith("r") and c.endswith("_rmse")], "RMSE of R (mm/h) by block size", 2)